In [13]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.1 MB/s eta 0:00:00


In [14]:
%%writefile quicksort_mpi.py
from mpi4py import MPI
import numpy as np

def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr)//2]
    left = arr[arr < pivot]
    mid = arr[arr == pivot]
    right = arr[arr > pivot]
    return np.concatenate([quicksort(left), mid, quicksort(right)])

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

if rank == 0:
    data = np.array([64, 34, 25, 12, 22, 11, 90], dtype='i')
    chunks = np.array_split(data, size)
else:
    chunks = None

local_data = comm.scatter(chunks, root=0)
local_sorted = quicksort(local_data)
gathered = comm.gather(local_sorted, root=0)

if rank == 0:
    result = quicksort(np.concatenate(gathered))
    print("Sorted array:", result)

Overwriting quicksort_mpi.py


In [15]:
!mpirun --allow-run-as-root --oversubscribe -n 4 python quicksort_mpi.py

Sorted array: [11 12 22 25 34 64 90]
